# NB04 — Building the First Modelling Table

**Role:** Core  
**Manual section:** 2.2.1 — Understand business and project objectives  
**Prerequisites:** NB03

---

## Purpose of this notebook

This notebook makes the transition from descriptive exploration to predictive work. You will define the business question, choose a unit of analysis and target variable, build a modelling table, and prepare a train/test split — all before any algorithm is called.

**Dataset:** `dataset_C_bank_attrition_core.csv` (10 000 customers × 14 columns) → saves `dataset_C_attrition_model_ready.csv`

## Section 0 — Why Modelling Starts Before the Model

In Notebooks 02 and 03 we learned to load, summarise and visualise data.  
Now comes the step that separates descriptive exploration from predictive work:
**building a modelling table**.

The hard part of a machine-learning project is almost never the `model.fit()` call.  
It is everything that comes before:

| Preparation step | Why it matters |
|-------------------|---------------|
| Define the business question | Without a question, the model has no purpose |
| Choose the unit of analysis | Determines what one row means |
| Select the target variable | Determines what the model tries to predict |
| Identify predictors vs identifiers | Prevents leakage and wasted information |
| Handle missing values | Garbage in → garbage out |
| Create a train/test split | Measures real-world performance, not memorisation |

> *"A bad modelling table can make a good algorithm look smart for the wrong reasons."*

## Section 0 — Project Framing Before Code

Before we write `import pandas as pd`, we must define the attrition project in business language.

### Six framing questions for this notebook

1. **What decision are we trying to improve?**  
   We want to help the bank decide **which customers should be prioritised for retention action**.

2. **Who will use the output?**  
   A customer-retention or CRM team that can call, message or target customers at risk.

3. **What is the unit of analysis?**  
   **One row = one customer.**

4. **What is the target?**  
   `attrition_flag`  
   - `1` = the customer leaves (attrition; also called churn)  
   - `0` = the customer stays

5. **What is the prediction horizon?**  
   For teaching purposes, interpret this as a **next-3-month attrition risk** problem.

6. **What is the key modelling risk at this stage?**  
   **Leakage**: using variables that would not truly be available when the bank makes the decision.

> **Core project notebook**  
> This notebook starts the running attrition project that continues through **NB04b → NB05 → NB05b → NB06**.

> **Optional short supports (10–15 min each)**  
> - **NB04a** for data structures in finance  
> - **NB04b** for treatment and validation of the shared modelling-table file

---

## Section 1 — From Business Question to Analytical Question

### The business problem

A retail bank has observed that roughly **one in five** customers eventually closes their
account. The bank wants to **identify customers at risk of attrition** so it can
launch a targeted retention campaign.

### Rewriting as an analytical question

> *Given a customer's demographic and banking profile, can we predict whether they will
> leave the bank (attrition_flag = 1) or stay (attrition_flag = 0)?*

This is a **binary classification** problem: the outcome has exactly two possible values.

### Why it matters for the business

| Prediction outcome | Business meaning |
|--------------------|-----------------|
| **True Positive** — correctly predicted attrition | Retention team can intervene early |
| **False Negative** — missed attrition case | Customer leaves without any action |
| **False Positive** — flagged a loyal customer | Unnecessary retention offer (small cost) |
| **True Negative** — correctly identified stayer | No action needed |

Missing a true attrition case is generally **more costly** than sending a retention offer to
someone who would have stayed anyway. We will revisit this trade-off when we discuss
evaluation metrics in Notebook 05.

---

## Section 2 — Define the Unit of Analysis

### What is one row?

Each row in our dataset represents **one customer** at a single point in time.

This means:
- There is no time series in this table (no repeated observations per customer).
- Each customer appears exactly once.
- The target (`attrition_flag`) records whether the customer eventually left.

### Why this matters

If rows were duplicated, or if a single customer appeared multiple times with different
outcomes, our model would learn from a distorted view of reality.

Let's verify this by loading the dataset:

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Resolve data directory — works from the project root or notebooks/
_candidates = [Path("data/processed"), Path("../data/processed")]
DATA_DIR = next((p for p in _candidates if p.is_dir()), None)
if DATA_DIR is None:
    raise FileNotFoundError(
        "Cannot locate data/processed/. "
        "Launch the notebook from the project root or the notebooks/ folder."
    )

df = pd.read_csv(DATA_DIR / "dataset_C_bank_attrition_core.csv")

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Unique customer IDs: {df['customer_id'].nunique():,}")
print(f"Duplicated rows: {df.duplicated().sum()}")

In [ ]:
df.head()

> **Confirmed:** 10 000 rows, 10 000 unique customer IDs, no duplicates.  
> One row = one customer.

---

## Section 3 — Target, Predictors and Identifiers

Not every column in the dataset should be used in a model.  
We need to classify each column by its **role**.

### Role classification

| Role | Definition | Rule |
|------|-----------|------|
| **Target** | What the model predicts | Exactly one column |
| **Predictor (feature)** | Information available *before* the outcome is known | Used for training |
| **Identifier** | A row label with no predictive signal | Must be dropped |
| **Leakage risk** | A column that encodes or correlates with the outcome in ways that would not be available at prediction time | Must be investigated |

In [ ]:
# Display all columns with their data types
print(f"{'Column':<25s} {'Dtype':<12s}")
print("-" * 37)
for col in df.columns:
    print(f"{col:<25s} {str(df[col].dtype):<12s}")

Let's map each column to a role:

In [ ]:
role_table = pd.DataFrame({
    "column": df.columns,
    "role": [
        "Identifier",       # customer_id
        "Predictor",        # credit_score
        "Predictor",        # geography
        "Predictor",        # gender
        "Predictor",        # age
        "Predictor",        # tenure
        "Predictor",        # balance
        "Predictor",        # num_of_products
        "Predictor",        # has_credit_card
        "Predictor",        # is_active_member
        "Predictor",        # estimated_salary
        "Target",           # attrition_flag
        "Predictor",        # digital_usage_proxy
        "Predictor",        # complaints_last_6m
    ],
    "note": [
        "Unique ID — no signal, must drop",
        "Numeric — financial health indicator",
        "Categorical — country",
        "Categorical — demographic",
        "Numeric — demographic",
        "Numeric — years as customer",
        "Numeric — account balance (EUR)",
        "Numeric — number of bank products held",
        "Binary — 1 if customer has a credit card",
        "Binary — 1 if customer is active",
        "Numeric — estimated annual salary (EUR)",
        "Binary — 1 = churned, 0 = stayed",
        "Numeric — derived teaching feature (digital engagement score)",
        "Numeric — derived teaching feature (complaint count, last 6 months)",
    ],
})
role_table

### Key observations

- **`customer_id`** is an identifier — it has no business meaning for prediction and must
  be removed.
- **`attrition_flag`** is our target.
- All other columns are candidate predictors.
- **`digital_usage_proxy`** and **`complaints_last_6m`** are *derived teaching features*
  added during dataset preparation. They are safe to use as predictors because they were
  built from information available before the outcome.

### Data leakage awareness

**Data leakage** occurs when the model has access to information that would not be
available at the time of prediction. Common forms:

| Leakage type | Example |
|-------------|---------|
| Target leakage | Using a column that is derived from the target itself |
| Temporal leakage | Using future information to predict a past event |
| Identifier leakage | A unique ID that accidentally correlates with the target |

Let's check that `customer_id` does not accidentally correlate with the target:

In [ ]:
corr_id = df["customer_id"].corr(df["attrition_flag"])
print(f"Correlation between customer_id and attrition_flag: {corr_id:.4f}")
print("→ Negligible. Safe to drop as expected.")

> **Leakage verdict:** no columns in this dataset encode or leak the target.  
> The identifiers have negligible correlation. We can proceed safely.

**Reflection questions:**
1. Could `is_active_member` be considered leakage? (Think: is activity status known
   *before* the customer decides to leave?)
2. If we had a column called `exit_date`, should it be a predictor?

---

## Section 4 — Missing Values and Basic Cleaning

Before building the modelling table, we must handle any missing or suspicious values.

In [ ]:
# Check missing values
missing = df.isna().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.sum() > 0 else "  None — all columns are complete.")

In [ ]:
# Quick sanity checks
print("Value ranges for key numeric columns:")
for col in ["credit_score", "age", "tenure", "balance", "estimated_salary"]:
    print(f"  {col:<22s} min={df[col].min():>10,.1f}  max={df[col].max():>10,.1f}")

No missing values and no impossible ranges detected.

In other datasets you may need to:
- **Fill** missing numeric values with the median.
- **Fill** missing categorical values with `"Unknown"` or the mode.
- **Drop** rows only if there is a clear reason (e.g., completely empty records).

For this dataset, **no imputation is needed** — we can move directly to encoding.

---

## Section 5 — Basic Transformations Before Modelling

Machine-learning models (including logistic regression and tree-based methods) typically
require **numeric inputs**. We need to convert categorical columns into numbers.

### Encoding strategy

| Column | Type | Encoding |
|--------|------|----------|
| `geography` | 3 categories (France, Germany, Spain) | One-hot encoding (dummy variables) |
| `gender` | 2 categories (Male, Female) | One-hot encoding (drop one to avoid redundancy) |

### Why one-hot encoding?

A categorical variable like `geography` has no natural numeric order.  
Assigning France = 1, Germany = 2, Spain = 3 would imply that Spain > Germany > France,
which is meaningless. One-hot encoding creates separate binary columns instead.

In [ ]:
# One-hot encode categorical columns
df_encoded = pd.get_dummies(df, columns=["geography", "gender"], drop_first=True)

# Show the new columns
new_cols = [c for c in df_encoded.columns if c not in df.columns]
print("New columns after encoding:", new_cols)
print(f"Shape: {df_encoded.shape[0]:,} rows × {df_encoded.shape[1]} columns")

In [ ]:
df_encoded.head()

> **What happened:**
> - `geography` became `geography_Germany` and `geography_Spain` (France is the reference).
> - `gender` became `gender_Male` (Female is the reference).
> - The original text columns are gone; only numeric columns remain.
>
> We used `drop_first=True` to avoid **perfect multicollinearity** — if a customer is not
> in Germany and not in Spain, they must be in France.

---

## Section 6 — The First Modelling Table

Now we separate the data into:
- **X** — the feature matrix (all predictors),
- **y** — the target vector (what we want to predict).

We also remove the identifier `customer_id`.

In [ ]:
# Define target and features
target_col = "attrition_flag"
id_col = "customer_id"

y = df_encoded[target_col]
X = df_encoded.drop(columns=[target_col, id_col])

print(f"Feature matrix X: {X.shape[0]:,} rows × {X.shape[1]} columns")
print(f"Target vector y:  {y.shape[0]:,} values")
print(f"Target distribution:\n{y.value_counts()}")

In [ ]:
# Final check: no identifiers, no target in X
print("Columns in X:")
print(list(X.columns))

In [ ]:
# Preview the modelling table
X.head()

> **Verified:**
> - `customer_id` is removed.
> - `attrition_flag` is removed from X and stored separately in y.
> - All remaining columns are numeric and ready for modelling.
> - 13 predictor columns.

---

## Section 7 — Train / Test Split

### Why split the data?

If we train and evaluate a model on the **same** data, we cannot know whether it has
learned genuine patterns or simply memorised the training examples.

A **train/test split** reserves a portion of the data (typically 20–30 %) that the model
**never sees during training**. We evaluate only on this held-out test set.

### Stratification

Because attrition is relatively rare (~20 %), we use **stratified splitting** to ensure that
the attrition rate is approximately the same in both the training and test sets.

### Save the canonical project dataset

We now save the **shared modelling table** that will be reused in later notebooks.

This is important for the learning design: students should feel that NB04, NB04b, NB05 and NB05b are not isolated exercises, but stages of **one real project**.


In [ ]:
# Build and save the canonical model-ready table used by later notebooks
model_ready = pd.concat([df_encoded[[id_col]], X, y], axis=1)

OUTPUT_PATH = DATA_DIR / 'dataset_C_attrition_model_ready.csv'
model_ready.to_csv(OUTPUT_PATH, index=False)

print(f'Saved shared project dataset to: {OUTPUT_PATH}')
print(f'Shape: {model_ready.shape[0]:,} rows × {model_ready.shape[1]} columns')
model_ready.head()

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print(f"Training set: {X_train.shape[0]:,} rows")
print(f"Test set:     {X_test.shape[0]:,} rows")
print()
print("Churn rate in training set:", f"{y_train.mean():.3f}")
print("Churn rate in test set:    ", f"{y_test.mean():.3f}")

> Both sets have approximately the same attrition rate (~20 %). The split is balanced.

**Why is this a discipline rather than a technicality?**

- It prevents **overfitting** — a model that works only on data it has already seen.
- It gives a realistic estimate of how the model would perform on **new customers**.
- It is the foundation of **trustworthy evaluation** in any ML project.

---

## Section 8 — Modelling-Table Checklist

Before moving on to actual model fitting, let's verify everything is in order.

### Pre-modelling checklist

| Check | Status |
|-------|--------|
| Business question clearly defined? | ✅ Predict customer attrition |
| Unit of analysis clear? | ✅ One row = one customer |
| Target variable identified? | ✅ `attrition_flag` (binary) |
| Identifiers removed from features? | ✅ `customer_id` dropped |
| Leakage risk assessed? | ✅ No leakage detected |
| Missing values handled? | ✅ No missing values in this dataset |
| Categorical encoding applied? | ✅ One-hot for geography, gender |
| Train/test split done with stratification? | ✅ 80/20, stratified |

> Save this checklist mentally — you will use the same logic in every modelling project.

### Bridge to the next notebooks

- **NB04b** loads this exact file (`dataset_C_attrition_model_ready.csv`) and validates/treats it as the canonical project table.
- **NB05** loads the same file to fit the logistic-regression baseline.
- **NB05b** loads the same file again to compare tree-based methods and XGBoost against the baseline.


---

## Section 9 — Transition to Baseline Models

The modelling table is ready. But before jumping to a complex model, we need a
**baseline** — a simple reference that tells us:

- What performance can we achieve with a straightforward approach?
- Is the problem solvable at all from these features?
- What is the minimum level a more complex model must beat?

In **Notebook 05** we will:
1. See the difference between regression and classification.
2. Fit a **logistic regression** as the baseline classifier.
3. Learn to read basic evaluation metrics (accuracy, precision, recall, F1, AUC).
4. Compare training and test performance.

> *"Before trying sophisticated AI, we need a clear baseline and a trustworthy modelling
> table."*

---

### Self-practice tasks

1. Change the set of predictors: remove one column and add a brief justification.
2. Identify which column(s) could be a leakage risk if the dataset were different
   (e.g., if `is_active_member` were measured *after* the customer left).
3. Try a different simple missing-value rule on another dataset and reflect on whether
   it is reasonable.
4. Write a 3-sentence analyst memo explaining the modelling-table design to a non-technical
   stakeholder.

---

*End of Notebook 04 — Building the First Modelling Table*

---

**Project chain:** NB01 (setup) → NB02 (Python warmup) → NB03 (EDA) → **NB04 (modelling table)** → NB05 (baseline) → NB05b (trees) → NB06 (validate & interpret)  
**Current position:** NB04